In [ ]:
%pip install -q segmentation-models-pytorch albumentations
import json, random
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import segmentation_models_pytorch as smp
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# датасет лежит рядом с проектом: data/dataset (см. README)
DS = Path("data/dataset")
IMG_DIRS = [DS / "talc_annotation", DS / "photos" / "otalkovannaya",
            DS / "photos" / "ryadovaya", DS / "photos" / "trudnoobogatimaya"]

def find_img(stem):
    for d in IMG_DIRS:
        for ext in (".jpg", ".jpeg", ".png", ".bmp"):
            if (d / f"{stem}{ext}").exists():
                return d / f"{stem}{ext}"

PAIRS = {mp.stem: (str(find_img(mp.stem)), str(mp))
         for mp in sorted((DS / "talc_masks").glob("*.png")) if find_img(mp.stem)}
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{device}; пар фото+маска: {len(PAIRS)}")

In [ ]:
# сплит по образцам: фото одного шлифа не разъезжаются
def group_of(stem):
    parts = stem.split("_")
    if len(parts) > 1 and parts[1].split("-")[0].isdigit():
        return parts[1].split("-")[0]
    return stem

stems = sorted(PAIRS)
groups = {s: group_of(s) for s in stems}
uniq = sorted(set(groups.values()))
rng = np.random.RandomState(0)
val_groups = set(rng.choice(uniq, max(int(len(uniq) * 0.25), 3), replace=False))
train_stems = [s for s in stems if groups[s] not in val_groups]
val_stems = [s for s in stems if groups[s] in val_groups]
print(f"train {len(train_stems)} / val {len(val_stems)}")

In [ ]:
CROP = 768
# аугментации: масштаб 80-500 мкм, освещение/контраст/гамма, расфокус
train_aug = A.Compose([
    A.RandomScale(scale_limit=(-0.6, 1.0), p=0.7),
    A.PadIfNeeded(CROP, CROP, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomCrop(CROP, CROP),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.3),
    A.RandomBrightnessContrast(0.35, 0.35, p=0.9),
    A.RandomGamma((60, 160), p=0.5),
    A.HueSaturationValue(15, 40, 30, p=0.8),
    A.RGBShift(25, 25, 25, p=0.5),
    A.GaussianBlur(blur_limit=(3, 9), p=0.3),
    A.GaussNoise(p=0.3),
])

class TalcDS(Dataset):
    def __init__(self, stems, aug=None, crops_per_image=16):
        self.items = [(s, i) for s in stems
                      for i in range(crops_per_image if aug else 1)]
        self.aug, self.cache = aug, {}
    def _load(self, stem):
        if stem not in self.cache:
            ip, mp = PAIRS[stem]
            img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
            m = (cv2.imread(mp, 0) > 127).astype(np.float32)
            self.cache[stem] = (img, m)
        return self.cache[stem]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        stem, _ = self.items[i]
        img, m = self._load(stem)
        if self.aug:
            r = self.aug(image=img, mask=m)
            img, m = r["image"], r["mask"]
        else:
            H = (img.shape[0] // 32) * 32; W = (img.shape[1] // 32) * 32
            img, m = img[:H, :W], m[:H, :W]
        x = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.
        return x, torch.from_numpy(np.ascontiguousarray(m)).float()

train_dl = DataLoader(TalcDS(train_stems, train_aug), batch_size=16,
                      shuffle=True, num_workers=4)
val_ds = TalcDS(val_stems)

In [ ]:
# resnet18 — как в src/talc.py::load_unet
model = smp.Unet("resnet18", encoder_weights="imagenet", classes=1).to(device)
dice = smp.losses.DiceLoss(mode="binary")
bce = nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
EPOCHS = 60
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

def val_mae():
    model.eval()
    errs, ious = [], []
    with torch.no_grad():
        for x, m in DataLoader(val_ds, batch_size=1):
            p = torch.sigmoid(model(x.to(device)))[0, 0].cpu().numpy()
            gt = m[0].numpy()
            errs.append(abs(p.mean() - gt.mean()))
            pb, gb = p > 0.5, gt > 0.5
            u = (pb | gb).sum()
            ious.append((pb & gb).sum() / u if u else 1.0)
    model.train()
    return float(np.mean(errs)), float(np.median(ious))

best, log = 1e9, []
for ep in range(EPOCHS):
    tot = 0.0
    for x, m in train_dl:
        x, m = x.to(device), m.to(device).unsqueeze(1)
        loss = 0.5 * bce(model(x), m) + 0.5 * dice(model(x), m)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += float(loss) * len(x)
    sched.step()
    mae, iou = val_mae()
    star = ""
    if mae < best:
        best = mae
        torch.save(model.state_dict(), "talc_unet.pt")
        star = "  <-- saved"
    print(f"ep {ep:02d} loss {tot/len(train_dl.dataset):.3f} "
          f"MAE {mae*100:.2f}пп IoU {iou:.2f}{star}", flush=True)
    log.append(dict(epoch=ep, loss=tot/len(train_dl.dataset), mae=mae, iou=iou))
json.dump(dict(best_mae=best, epochs=EPOCHS, arch="unet_resnet18", crop=CROP, log=log),
          open("unet_metrics.json", "w"), ensure_ascii=False, indent=2)
# результат: talc_unet.pt -> artifacts/talc/unet.pt
print(f"best MAE {best*100:.2f} пп")